In [ ]:
!pip install transformers torch datasets matplotlib seaborn scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict


# HuggingFace — only for backbone encoder, not for reward modeling logic
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
)

# Reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


# Why Do We Need Reward Models?

- After SFT, a model can follow instructions — but it has no sense of which of its possible outputs is best.
- Given a math problem, it might generate ten different solutions: some correct, some wrong, some with flawed reasoning that happens to reach the right answer.
- SFT alone can't distinguish these because it only learned to mimic training data, not to evaluate quality.
- GRPO doesn't replace reward models — they do different jobs. GRPO is the optimization algorithm (how to update weights), ORM/PRM are the scoring functions (how good is each output). GRPO generates multiple responses and reinforces the better ones, but it needs scores to rank them. For math, a simple rule works (check the answer). For open-ended tasks like summarization, you need a learned reward model to provide those scores.
- PRM also gives something GRPO can't: step-level feedback. GRPO only compares full responses, but a PRM scores each reasoning step individually — pinpointing exactly where the logic broke instead of penalizing the entire response.



In reinforcement learning for LLMs, the central challenge is:
> **Which tokens / steps are responsible for a correct or incorrect answer?**

Consider a multi-step math solution with 8 steps where step 3 is wrong.

![](https://www.marktechpost.com/wp-content/uploads/2025/01/Screenshot-2025-01-19-at-11.30.26%E2%80%AFAM.png)


Process Reward Models (PRMs) and Outcome Reward Models (ORMs) differ in how they evaluate LLM generations:

**Outcome Reward Model (ORM)**:
- Looks only at the final answer
- Assigns reward = 1 if correct, 0 if wrong
- ALL 8 steps get the same sparse signal
- Cannot identify *where* the error occurred

**Process Reward Model (PRM)**:
- Scores each individual step
- Step 3 gets a low score; steps 1,2 get high scores
- Credit is correctly assigned — dense signal

# 2) Outcome Reward Model (ORM)

An ORM maps a (problem, full_solution) pair to a scalar reward.
It only evaluates the quality of the *outcome* — not the reasoning process.

**Architecture:**
1. Load a pretrained LM backbone (e.g., DistilBERT, LLaMA, GPT-2)
2. Remove the original language modeling head
3. Add a scalar regression head on top of the [EOS] / last token
4. Train with Bradley-Terry loss (pairwise) or BCE loss (binary)

**Training Objective: Bradley-Terry Model**
The Bradley-Terry (BT) model is the foundation of all pairwise reward modeling.
Given two solutions y_w (winner/correct) and y_l (loser/incorrect):

$$
    P(y_w ≻ y_l | x) = σ(r(x, y_w) − r(x, y_l))
$$

$$
    Loss_BT = −log P(y_w ≻ y_l | x)
            = −log σ(r(x, y_w) − r(x, y_l))
$$

This is a **pairwise ranking loss** — we only care that correct solutions score higher than incorrect ones, not their absolute values.

### Alternative: Binary Cross-Entropy (BCE)
If we have binary labels (correct=1, incorrect=0):
$$
    Loss_BCE = −[y * log(σ(r)) + (1−y) * log(1 − σ(r))]
$$
Both approaches are used. BT is standard for RLHF (preference pairs).

BCE is common when you have verified correct/incorrect labels (math problems).

In [ ]:
def load_backbone(model_name: str = "distilbert-base-uncased"):
    """
    Load a pretrained transformer backbone and its tokenizer.
    """
    print(f"Loading backbone: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    backbone = AutoModel.from_pretrained(model_name)
    hidden_dim = backbone.config.hidden_size

    # Add pad token if not present
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"  hidden_dim = {hidden_dim}")
    print(f"  vocab_size  = {tokenizer.vocab_size}")
    return tokenizer, backbone, hidden_dim

tokenizer, backbone, HIDDEN_DIM = load_backbone("distilbert-base-uncased")
backbone = backbone.to(device)

## 2.1 Architecture

The Outcome Reward Model takes a complete (problem, solution) pair and outputs a single scalar reward score. It is used to judge whether a full reasoning chain is correct.

Architecture:
1. `[CLS] problem [SEP] solution_step_1 ... solution_step_K [EOS]`
2. Transformer backbone (frozen or fine-tuned)
3. Hidden state at `[CLS]` / last token position
4. Scalar regression head (2-layer MLP)
5. Reward ∈ ℝ (unbounded; higher = more correct)

In [ ]:
# === Train a Reward Model ===
class OutcomeRewardModel(nn.Module):
    """
    Outcome Reward Model (ORM).
    Takes a complete (problem, solution) pair and outputs a single scalar reward.
    """
    def __init__(self, backbone: nn.Module, hidden_dim: int, dropout_p: float = 0.1):
        super().__init__()
        self.backbone = backbone

        # Scalar reward head: 2-layer MLP with GELU activation
        # (batch_num, hidden_dim) → (batch_num, 1)
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim // 2, 1),
        )

        # Initialize reward head weights to small values
        nn.init.normal_(self.reward_head[0].weight, std=0.02)
        nn.init.zeros_(self.reward_head[0].bias)
        nn.init.normal_(self.reward_head[-1].weight, std=0.02)
        nn.init.zeros_(self.reward_head[-1].bias)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: encode (problem + solution) → scalar reward.
        """

        # (batch_num, seq_len) → (batch_num, seq_len, hidden_dim)
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state

        # Extract representation from the [CLS] token (position 0)
        # (batch_num, seq_len, hidden_dim) → (batch_num, hidden_dim)
        cls_repr = last_hidden[:, 0, :]

        # Pass CLS representation through reward head
        # (batch_num, hidden_dim) → (batch_num, 1)
        reward_logit = self.reward_head(cls_repr)

        # Squeeze last dim → scalar reward per example
        # (batch_num, 1) → (batch_num,)
        return reward_logit.squeeze(-1)

orm = OutcomeRewardModel(backbone, HIDDEN_DIM).to(device)
orm_params = sum(p.numel() for p in orm.reward_head.parameters())
print(f"\nORM reward head params: {orm_params:,}")
print(f"ORM total params: {sum(p.numel() for p in orm.parameters()):,}")


ORM reward head params: 295,681
ORM total params: 66,658,561


## 2.2 Loss Functions
**Bradley-Terry Pairwise Loss** (standard RLHF):
$$
    L_BT = −log σ(r(x, y_chosen) − r(x, y_rejected))
$$

**Binary Cross-Entropy Loss** (for verified labels):
$$
    L_BCE = −[y * log(σ(r)) + (1−y) * log(1−σ(r))]
$$


**Margin-Ranking Loss** (optional regularization):
$$
    L_margin = max(0, m − (r_chosen − r_rejected))
$$

In [ ]:
def bradley_terry_loss(r_chosen: torch.Tensor, r_rejected: torch.Tensor) -> torch.Tensor:
    """
    Bradley-Terry pairwise ranking loss.
    L = log(1 + exp(r_r − r_c)) = softplus(r_r − r_c)
    """
    # (batch_num,) and (batch_num,) → (batch_num,)
    margin = r_chosen - r_rejected

    # (batch_num,) → (batch_num,)
    loss = F.softplus(-margin)

    # (batch_num,) → scalar
    return loss.mean()

def bce_reward_loss(rewards: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Binary Cross-Entropy loss for ORM with verified labels.
    """
    # (batch_num,) and (batch_num,) → scalar
    return F.binary_cross_entropy_with_logits(rewards, labels)

def margin_ranking_loss(r_chosen: torch.Tensor, r_rejected: torch.Tensor, margin: float = 0.5):
    """
    Margin ranking loss: enforces r_chosen > r_rejected by at least margin.
    """
    # (batch_num,) → (batch_num,)
    violation = margin - (r_chosen - r_rejected)

    # (batch_num,) → (batch_num,)
    loss = torch.clamp(violation, min=0.0)
    return loss.mean()

## 2.3 Training Dataset

Each training example is a **preference pair**:
    `(problem, chosen_solution, rejected_solution)`

Where:
- `chosen_solution`: a solution that arrives at the CORRECT final answer
- `rejected_solution`: a solution that arrives at the WRONG final answer

The ORM never sees intermediate steps — it only sees the complete solution.

In [ ]:
@dataclass
class ORMExample:
    problem: str
    chosen_solution: str
    rejected_solution: str
    correct_answer: str

ORM_TRAINING_DATA: List[ORMExample] = [
    ORMExample(
        problem="A store had 50 apples. They sold 30% on Monday and 20% of the remainder on Tuesday. How many apples are left?",
        chosen_solution="Step 1: Monday sales = 30% of 50 = 15. Step 2: After Monday: 50 − 15 = 35 apples. Step 3: Tuesday sales = 20% of 35 = 7. Step 4: After Tuesday: 35 − 7 = 28 apples. The answer is 28.",
        rejected_solution="Step 1: Monday sales = 30% of 50 = 15. Step 2: Tuesday sales = 20% of 50 = 10. Step 3: Total sold = 15 + 10 = 25. Step 4: Remaining = 50 − 25 = 25 apples. The answer is 25.",
        correct_answer="28"
    ),
    # ... (other examples omitted for brevity in minimal run, add them back in full run)
]

class ORMDataset(torch.utils.data.Dataset):
    def __init__(self, examples: List[ORMExample], tokenizer, max_len: int = 512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_len = max_len

    def _encode(self, problem: str, solution: str) -> Dict:
        # Encode as sentence pair
        encoded = self.tokenizer(
            problem, solution, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in encoded.items()}

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx: int) -> Dict:
        ex = self.examples[idx]
        chosen = self._encode(ex.problem, ex.chosen_solution)
        rejected = self._encode(ex.problem, ex.rejected_solution)

        return {
            "chosen_input_ids": chosen["input_ids"],
            "chosen_attention_mask": chosen["attention_mask"],
            "rejected_input_ids": rejected["input_ids"],
            "rejected_attention_mask": rejected["attention_mask"],
        }

orm_dataset = ORMDataset(ORM_TRAINING_DATA, tokenizer)
orm_loader = torch.utils.data.DataLoader(orm_dataset, batch_size=2, shuffle=True)
print(f"\nORM dataset: {len(orm_dataset)} preference pairs")


ORM dataset: 1 preference pairs


## 2.4 Training

In [ ]:
def train_orm(
    model: OutcomeRewardModel,
    loader: torch.utils.data.DataLoader,
    num_epochs: int = 2, lr: float = 2e-5,
    margin: float = 0.0
) -> Tuple[List[float], List[float]]:

    # Freeze backbone
    for param in model.backbone.parameters():
        param.requires_grad = False

    optimizer = torch.optim.AdamW(model.reward_head.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    model.train()
    loss_history = []
    accuracy_history = []

    for epoch in range(num_epochs):
        epoch_losses = []
        epoch_correct = 0
        epoch_total = 0

        for batch in loader:
            c_ids = batch["chosen_input_ids"].to(device)
            c_mask = batch["chosen_attention_mask"].to(device)
            r_ids = batch["rejected_input_ids"].to(device)
            r_mask = batch["rejected_attention_mask"].to(device)

            # (batch_num, seq_len) → (batch_num,)
            r_chosen = model(c_ids, c_mask)
            r_rejected = model(r_ids, r_mask)

            bt_loss = bradley_terry_loss(r_chosen, r_rejected)
            total_loss = bt_loss
            if margin > 0:
                total_loss = bt_loss + 0.1 * margin_ranking_loss(r_chosen, r_rejected, margin=margin)

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.reward_head.parameters(), 1.0)
            optimizer.step()

            epoch_losses.append(total_loss.item())
            correct = (r_chosen > r_rejected).sum().item()
            epoch_correct += correct
            epoch_total += c_ids.size(0)

        scheduler.step()
        mean_loss = sum(epoch_losses) / len(epoch_losses)
        accuracy = epoch_correct / epoch_total
        loss_history.append(mean_loss)
        accuracy_history.append(accuracy)

        print(f"  Epoch {epoch+1:3d}/{num_epochs} | BT Loss: {mean_loss:.4f} | Ranking Acc: {accuracy:.2%}")

    return loss_history, accuracy_history

# === Test Run ===
orm_losses, orm_accs = train_orm(orm, orm_loader, num_epochs=2, lr=2e-4)


TRAINING: Outcome Reward Model (Bradley-Terry)
  Epoch   1/2 | BT Loss: 0.7073 | Ranking Acc: 0.00%
  Epoch   2/2 | BT Loss: 0.6946 | Ranking Acc: 0.00%


## 2.5 Inference

### ORM at Inference Time: Best-of-N (BoN) Selection

The most common use of an ORM at inference time is **Best-of-N**:
1. Sample N solutions from a policy LM for a given problem
2. Score each with the ORM: `r_i = ORM(problem, solution_i)`
3. Return the solution with the highest ORM score

This is also called **reranking** or **weighted voting**.

Formally:
$$
    ŷ = argmax_{i ∈ [N]} r_ORM(x, y_i)
$$

Best-of-N compute scales as O(N) — each solution needs one ORM forward pass.

In [ ]:
@torch.no_grad()
def orm_score(
        model: OutcomeRewardModel,
        problem: str,
        solution: str,
        tokenizer,
        max_len: int = 512
    ) -> float:

    """
    Score a single (problem, solution) pair with the ORM.
    """
    model.eval()

    # Tokenize (problem, solution) pair
    # Output: dict with (1, seq_len) tensors
    encoded = tokenizer(
        problem, solution, return_tensors="pt", truncation=True,
        max_length=max_len, padding="max_length"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # Forward: (1, seq_len) → scalar
    score = model(input_ids, attention_mask)

    return score.item()

def best_of_n_orm(
        model: OutcomeRewardModel,
        problem: str,
        solutions: List[str],
        tokenizer, verbose: bool = True
    ) -> Tuple[str, float, List[float]]:

    """
    Best-of-N selection using ORM scoring.
    """
    all_scores = []
    for i, sol in enumerate(solutions):
        # Score each solution
        score = orm_score(model, problem, sol, tokenizer)
        all_scores.append(score)
        if verbose:
            print(f"  Solution {i+1}: score={score:+.4f} | snippet={sol[:60]!r}")

    # Select best solution by score
    best_idx = int(np.argmax(all_scores))
    best_solution = solutions[best_idx]
    best_score = all_scores[best_idx]

    return best_solution, best_score, all_scores

# Demo ORM inference
test_problem = "A store had 50 apples. They sold 30% Monday and 20% of remainder Tuesday. How many left?"
candidate_solutions = [
    "30% of 50 = 15. After Monday: 35. 20% of 35 = 7. After Tuesday: 28. Answer: 28.",
    "30% of 50 = 15. Tuesday: 20% of 50 = 10. Total sold: 25. Remaining: 25. Answer: 25.",
    "Sold 30% then 20% so total = 50% sold. 50 × 0.5 = 25 remaining. Answer: 25.",
    "Monday: 15 sold, 35 left. Tuesday: 7 sold, 28 left. Answer: 28.",
]
best_sol, best_score, scores = best_of_n_orm(orm, test_problem, candidate_solutions, tokenizer)
print(f"\nBest solution (score={best_score:+.4f}):\n  {best_sol}")


DEMO: ORM Best-of-N Selection
  Solution 1: score=-0.0277 | snippet='30% of 50 = 15. After Monday: 35. 20% of 35 = 7. After Tuesd'
  Solution 2: score=-0.0365 | snippet='30% of 50 = 15. Tuesday: 20% of 50 = 10. Total sold: 25. Rem'
  Solution 3: score=-0.0374 | snippet='Sold 30% then 20% so total = 50% sold. 50 × 0.5 = 25 remaini'
  Solution 4: score=-0.0337 | snippet='Monday: 15 sold, 35 left. Tuesday: 7 sold, 28 left. Answer: '

Best solution (score=-0.0277):
  30% of 50 = 15. After Monday: 35. 20% of 35 = 7. After Tuesday: 28. Answer: 28.


# 3) Process Reward Model (PRM)

**What is a PRM?**

A PRM assigns a reward to **each individual reasoning step** in a solution.
Instead of waiting until the end, it gives dense, step-level feedback.

**Architecture Difference from ORM**

Both ORM and PRM use the same type of transformer backbone, but differ in:
1. **Input**: PRM takes (problem + steps up to step k) — partial solution
2. **Output**: PRM outputs one score per step, not one per solution
3. **Loss**: PRM uses BCE at each step position independently
4. **Label**: Each step has its own {correct=1, incorrect=0} label

**Step Token Format**

```
Problem: A store...
Step 1: Monday sales = 15. ▪ [LABEL: 1 = correct]
Step 2: After Monday: 35.  ▪ [LABEL: 1 = correct]
Step 3: Tuesday sales = 10. ▪ [LABEL: 0 = WRONG — should be 7]
Step 4: Remaining = 25.    ▪ [LABEL: 0 = wrong]
```

The ▪ token (step separator) is where predictions are made.
The model learns to predict step correctness from this position.

## 3.1 Reward Model

In [ ]:
class ProcessRewardModel(nn.Module):
    """
    Process Reward Model (PRM).
    Assigns a correctness probability to each reasoning step.
    """
    def __init__(self, backbone: nn.Module, hidden_dim: int, dropout_p: float = 0.1):
        super().__init__()
        self.backbone = backbone

        # Step-level reward model scoring head: shared across all step positions
        # (num_steps, hidden_dim) → (num_steps, 1)
        self.step_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim // 2, 1),
        )

        for module in self.step_head:
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, std=0.02)
                nn.init.zeros_(module.bias)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        step_token_positions: torch.Tensor
    ) -> torch.Tensor:
        """
        Forward pass: encode partial solution → step-level reward scores.
        """
        # (batch_num, seq_len) → (batch_num, seq_len, hidden_dim)
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state

        B = input_ids.size(0)          # batch_num
        K = step_token_positions.size(1)  # num_steps

        # Gather hidden states at each [STEP] token position
        pos_expanded = step_token_positions.unsqueeze(-1).expand(B, K, last_hidden.size(-1))
        # Input: (batch_num, seq_len, hidden_dim) and (batch_num, num_steps, hidden_dim)
        # Output: (batch_num, num_steps, hidden_dim)
        step_hidden = torch.gather(last_hidden, dim=1, index=pos_expanded)

        # Reshape for shared step head
        # (batch_num, num_steps, hidden_dim) → (batch_num * num_steps, hidden_dim)
        # take this as CLS for each step
        step_hidden_flat = step_hidden.view(B * K, -1)

        # Apply shared step scoring head
        # (batch_num * num_steps, hidden_dim) → (batch_num * num_steps, 1)
        step_scores_flat = self.step_head(step_hidden_flat)

        # Reshape back
        # (batch_num * num_steps, 1) → (batch_num, num_steps)
        step_scores = step_scores_flat.view(B, K)

        return step_scores

# Instantiate PRM
prm = ProcessRewardModel(backbone, HIDDEN_DIM).to(device)
prm_head_params = sum(p.numel() for p in prm.step_head.parameters())
print(f"\nPRM step head params: {prm_head_params:,}")


PRM step head params: 295,681


## 3.2 Loss Function

The PRM predicts a binary correctness label for each step.
Loss is BCE at each step position, averaged over all steps.

$$
    L_PRM = − (1/K) Σ_{k=1}^{K} [y_k * log σ(r_k) + (1−y_k) * log(1−σ(r_k))]
$$

Where:
    y_k ∈ {0, 1}: label for step k (1=correct, 0=incorrect)
    r_k ∈ ℝ:      PRM logit for step k (before sigmoid)


**Important implementation detail:**

Steps after the FIRST incorrect step should also be labeled as incorrect, even if they are locally correct.

## 3.3 PRM Training Data

- The biggest challenge for PRMs is **obtaining step-level labels**.

- Three strategies exist, ordered by quality vs. cost:

```python
┌─────────────────────────┬───────────┬───────────┬──────────────┐
│ Strategy                │ Quality   │ Cost      │ Scalability  │
├─────────────────────────┼───────────┼───────────┼──────────────┤
│ Human annotation        │ Highest   │ Very high │ Low          │
│ (PRM800K)               │           │           │              │
├─────────────────────────┼───────────┼───────────┼──────────────┤
│ LLM-as-a-judge          │ High      │ Medium    │ Medium       │
│ (Qwen2.5-72B as judge)  │           │           │              │
├─────────────────────────┼───────────┼───────────┼──────────────┤
│ Monte Carlo estimation  │ Medium    │ Low       │ High         │
│ (Math-Shepherd)         │           │           │              │
└─────────────────────────┴───────────┴───────────┴──────────────┘
```

In [ ]:
@dataclass
class PRMStep:
    """A single reasoning step with its correctness label."""
    text: str
    label: int
    is_key: bool = False

@dataclass
class PRMExample:
    """
    A single PRM training example.
    Contains a problem and its full step-by-step solution, where each step has a correctness label.
    """
    problem: str
    steps: List[PRMStep]

    @property
    def num_steps(self) -> int:
        return len(self.steps)

    @property
    def labels(self) -> List[int]:
        return [s.label for s in self.steps]

    @property
    def is_correct_solution(self) -> bool:
        return all(s.label == 1 for s in self.steps)

## 3.4 Annotation Strategy

### 3.4.1 Human-Annotated (PRM800K style)

Lightman et al. (2023) hired human annotators to label each step as:
    - ✓ (correct): step is mathematically valid and logically sound
    - ✗ (incorrect): step contains an error
    - ? (neutral): step is valid but doesn't help

PRM800K dataset: ~800,000 step-level labels across ~75,000 problems.

Key insight: **supervise up to the first incorrect step only**.

Later steps are also labeled incorrect (contaminated by earlier error).

In [ ]:
HUMAN_ANNOTATED_EXAMPLES: List[PRMExample] = [
    PRMExample(
        problem="A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?",
        steps=[
            PRMStep("Monday sales = 30% of 50 = 0.30 × 50 = 15 apples.", label=1),
            PRMStep("After Monday: 50 − 15 = 35 apples remaining.", label=1),
            PRMStep("Tuesday sales = 20% of 35 = 0.20 × 35 = 7 apples.", label=1),
            PRMStep("After Tuesday: 35 − 7 = 28 apples left.", label=1),
        ]
    ),
    PRMExample(
        problem="A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?",
        steps=[
            PRMStep("Monday sales = 30% of 50 = 15.", label=1),
            # ERROR: should apply 20% to REMAINDER (35), not original 50
            PRMStep("Tuesday sales = 20% of 50 = 10.", label=0, is_key=True),
            PRMStep("Total sold = 15 + 10 = 25.", label=0),
            PRMStep("Remaining = 50 − 25 = 25 apples.", label=0),
        ]
    ),
    # (Other examples omitted for brevity in minimal run, add them back in full run)
]

print(f"Human-annotated PRM examples: {len(HUMAN_ANNOTATED_EXAMPLES)}")
print(f"  Correct solutions:   {sum(1 for e in HUMAN_ANNOTATED_EXAMPLES if e.is_correct_solution)}")
print(f"  Incorrect solutions: {sum(1 for e in HUMAN_ANNOTATED_EXAMPLES if not e.is_correct_solution)}")

Human-annotated PRM examples: 2
  Correct solutions:   1
  Incorrect solutions: 1


### 3.4.2: Monte Carlo Estimation (Math-Shepherd)

The key insight of Math-Shepherd (Wang et al. 2023):
> **A step is "good" if rolling out from it can still reach the correct answer.**

Process:
1. For a given step, imagine you try to complete the rest of the solution M times (e.g., 16 attempts) starting from that step.
2. If at least one completion reaches the correct final answer, the step is probably fine (hard_label=1).
3. If zero completions succeed, the step is likely wrong (hard_label=0).
4. The soft_label gives you a probability (e.g., 12/16 = 0.75).

In [ ]:
import openai
from openai import OpenAI
openai.api_key = ''
client = openai.OpenAI()

def extract_answer(text: str) -> str:
    """Extract the final numerical answer from a completion."""
    # Look for common patterns: "= 28", "is 28", "answer is 28", just "28" at the end
    patterns = [
        r"(?:answer|result|remaining|left|total)\s*(?:is|=|:)\s*([\d.,]+)",
        r"=\s*([\d.,]+)\s*$",
        r"([\d.,]+)\s*$",
    ]
    for pat in patterns:
        match = re.search(pat, text.strip(), re.IGNORECASE)
        if match:
            return match.group(1).replace(",", "").strip(".")
    return ""

In [ ]:
def mc_complete(
    problem: str,
    step_prefix: List[str],
    gold_answer: str,
    M: int = 10,
    model: str = "gpt-4o-mini",
) -> Tuple[int, float]:
    """
    Monte Carlo estimation: given a problem and partial steps,
    ask the LLM to complete the solution M times.
    Check how many completions reach the correct final answer.

    Returns:
        hard_label: 1 if at least one completion is correct, else 0
        soft_label: fraction of correct completions (successes / M)
    """

    prefix_text = "\n".join(step_prefix)
    prompt = (
        f"Problem: {problem}\n\n"
        f"Solution so far:\n{prefix_text}\n\n"
        f"Continue solving from where the solution left off. "
        f"Show your work step by step and end with the final numerical answer."
    )

    successes = 0

    # Generate M completions
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        n=M,                  # generate M completions in one API call
        temperature=0.7,      # some randomness for diverse completions
        max_tokens=200,
    )

    for choice in response.choices:
        completion = choice.message.content
        extracted = extract_answer(completion)
        if extracted == gold_answer:
            successes += 1

    hard_label = int(successes > 0)
    soft_label = successes / M
    return hard_label, soft_label


def auto_annotate_mc(
    problem: str,
    steps: List[str],
    gold_answer: str,
    M: int = 8,
) -> "PRMExample":
    """
    Walk through each step, run MC completions from that prefix.
    Once first error is found, mark all subsequent steps as wrong.
    """
    annotated_steps = []
    found_error = False

    for k, step_text in enumerate(steps):
        if found_error:
            annotated_steps.append(PRMStep(text=step_text, label=0))
            continue

        prefix = steps[: k + 1]
        hard_label, soft_label = mc_complete(problem, prefix, gold_answer, M=M)

        print(f"  Step {k+1}: MC success rate = {soft_label:.0%} | hard_label={hard_label}")

        if hard_label == 0:
            found_error = True
            annotated_steps.append(PRMStep(text=step_text, label=0, is_key=True))
        else:
            annotated_steps.append(PRMStep(text=step_text, label=1))

    return PRMExample(problem=problem, steps=annotated_steps)


mc_problem = "A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?"
correct_steps = [
    "Monday sales = 30% of 50 = 15.",
    "After Monday: 50 − 15 = 35 remaining.",
    "Tuesday sales = 20% of 35 = 7.",
    "After Tuesday: 35 − 7 = 28 left.",
]
wrong_steps = [
    "Monday sales = 30% of 50 = 15.",
    "Tuesday sales = 20% of 50 = 10.",
    "Total sold = 25.",
    "Remaining = 25.",
]

print("Annotating correct solution...")
mc_correct = auto_annotate_mc(mc_problem, correct_steps, "28", M=8)

print("\nAnnotating wrong solution...")
mc_wrong = auto_annotate_mc(mc_problem, wrong_steps, "28", M=8)

print("\n--- Correct solution MC labels ---")
for i, step in enumerate(mc_correct.steps):
    print(f"  Step {i+1}: label={step.label} | {step.text}")

print("\n--- Incorrect solution MC labels ---")
for i, step in enumerate(mc_wrong.steps):
    tag = " ← FIRST ERROR" if step.is_key else ""
    print(f"  Step {i+1}: label={step.label}{tag} | {step.text}")

### 3.4.3 Annotation Strategy 3: LLM-as-a-Judge

Use a strong LLM to evaluate each step.

Prompt the judge with the problem, prior steps, and the current step, and ask it to classify the step as correct or incorrect.

In [ ]:

LLM_JUDGE_PROMPT_TEMPLATE = """
You are a mathematics verification expert.

Problem: {problem}

Previous steps:
{prev_steps}

Current step to evaluate:
{current_step}

Is this step mathematically correct given the problem and previous steps?
Answer with only: CORRECT or INCORRECT
Reason: (brief explanation)
"""

def llm_judge(prompt: str, model: str = "gpt-4o-mini") -> str:
    """Call OpenAI API to judge whether a reasoning step is correct."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,  # deterministic for consistent judgments
        max_tokens=100,
    )
    return response.choices[0].message.content


def llm_judge_annotation(
    problem: str,
    steps: List[str],
    model: str = "gpt-4o-mini",
) -> "PRMExample":
    """
    Walk through each step and ask the LLM judge if it's correct.
    Once first error is found, mark all subsequent steps as wrong.
    """
    annotated = []
    found_error = False

    for k, step in enumerate(steps):
        if found_error:
            annotated.append(PRMStep(text=step, label=0))
            continue

        prev_steps_str = (
            "\n".join(f"Step {i+1}: {s}" for i, s in enumerate(steps[:k]))
            if k > 0
            else "(none)"
        )
        prompt = LLM_JUDGE_PROMPT_TEMPLATE.format(
            problem=problem,
            prev_steps=prev_steps_str,
            current_step=f"Step {k+1}: {step}",
        )

        verdict = llm_judge(prompt, model=model)
        label = 1 if "CORRECT" in verdict.strip().upper() else 0

        print(f"  Step {k+1}: {verdict.strip()}")

        if label == 0:
            found_error = True
            annotated.append(PRMStep(text=step, label=0, is_key=True))
        else:
            annotated.append(PRMStep(text=step, label=1))

    return PRMExample(problem=problem, steps=annotated)


# === Test Run ===
mc_problem = "A store had 50 apples. Sold 30% Monday and 20% of remainder Tuesday. How many left?"

correct_steps = [
    "Monday sales = 30% of 50 = 15.",
    "After Monday: 50 − 15 = 35 remaining.",
    "Tuesday sales = 20% of 35 = 7.",
    "After Tuesday: 35 − 7 = 28 left.",
]
wrong_steps = [
    "Monday sales = 30% of 50 = 15.",
    # ERROR: should be 20% of 35
    "Tuesday sales = 20% of 50 = 10.",
    "Total sold = 25.",
    "Remaining = 25.",
]

print("LLM Judge on correct solution:")
llm_correct = llm_judge_annotation(mc_problem, correct_steps)

print("\nLLM Judge on wrong solution:")
llm_wrong = llm_judge_annotation(mc_problem, wrong_steps)

print("\n--- Correct solution labels ---")
for i, step in enumerate(llm_correct.steps):
    print(f"  Step {i+1}: label={step.label} | {step.text}")

print("\n--- Incorrect solution labels ---")
for i, step in enumerate(llm_wrong.steps):
    tag = " ← FIRST ERROR" if step.is_key else ""
    print(f"  Step {i+1}: label={step.label}{tag} | {step.text}")


DEMO: LLM-as-a-Judge Annotation (mock judge)
LLM judge labels on wrong solution:
  Step 1: label=1  | Monday sales = 30% of 50 = 15.
  Step 2: label=1  | Tuesday sales = 20% of 50 = 10.
  Step 3: label=1  | Total sold = 25.
  Step 4: label=1  | Remaining = 25.


## 3.5 PRM Dataset and Tokenization

**The Step Token Trick**

The PRM needs to know WHERE in the token sequence each step ends.
The approach from Math-Shepherd and PRM800K:

1. Append a special [STEP] token after each step in the sequence
2. Record the token position of each [STEP] token
3. During forward pass: extract hidden state at [STEP] positions
4. Apply step head → step scores
5. Compute BCE loss at step positions ONLY (rest masked)

Input sequence:
    [CLS] problem text [SEP] step_1_text ▪ step_2_text ▪ step_3_text ▪
                                          ↑              ↑              ↑
                                       pos 15          pos 28         pos 41
                                    label=1          label=1         label=0

In [ ]:
STEP_SEPARATOR = " ▪"

class PRMDataset(torch.utils.data.Dataset):
    """
    Dataset for PRM training.
    """
    def __init__(self, examples: List, tokenizer, max_len: int = 512, max_steps: int = 8):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.max_steps = max_steps

        # Use '.' as a proxy step separator token (already in vocab)
        sep_tokens = tokenizer(" .", add_special_tokens=False)["input_ids"]
        self.step_token_id = sep_tokens[0] if sep_tokens else 1012

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx: int) -> Dict:
        ex = self.examples[idx]
        sequence_parts = [ex.problem]
        for step in ex.steps:
            sequence_parts.append(step.text + STEP_SEPARATOR)
        full_text = " ".join(sequence_parts)

        # Output: (seq_len,) for each tensor
        encoded = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)

        # Find positions of step separator tokens
        sep_id = self.step_token_id
        step_positions_list = (input_ids == sep_id).nonzero(as_tuple=True)[0].tolist()

        num_actual_steps = min(len(step_positions_list), ex.num_steps, self.max_steps)
        labels = ex.labels[:num_actual_steps]

        step_positions = torch.zeros(self.max_steps, dtype=torch.long)
        step_labels = torch.zeros(self.max_steps, dtype=torch.float)
        step_mask = torch.zeros(self.max_steps, dtype=torch.float)

        for i in range(num_actual_steps):
            if i < len(step_positions_list):
                pos = min(step_positions_list[i], self.max_len - 1)
                step_positions[i] = pos
                step_labels[i] = labels[i] if i < len(labels) else 0.0
                step_mask[i] = 1.0

        return {
            "input_ids": input_ids,               # (seq_len,)
            "attention_mask": attention_mask,     # (seq_len,)
            "step_positions": step_positions,     # (max_steps,)
            "step_labels": step_labels,           # (max_steps,)
            "step_mask": step_mask,               # (max_steps,)
        }

# Build combined dataset
all_prm_examples = HUMAN_ANNOTATED_EXAMPLES.copy()
all_prm_examples.append(auto_annotate_mc(mc_problem, correct_steps, "28", M=20))
all_prm_examples.append(auto_annotate_mc(mc_problem, wrong_steps, "28", M=20))

prm_dataset = PRMDataset(all_prm_examples, tokenizer)
prm_loader = torch.utils.data.DataLoader(prm_dataset, batch_size=2, shuffle=True)
print(f"\nPRM dataset: {len(prm_dataset)} examples")


PRM dataset: 4 examples


## 3.6 Training


In [ ]:
def train_prm(
        model: ProcessRewardModel,
        loader: torch.utils.data.DataLoader,
        num_epochs: int = 2,
        lr: float = 2e-4
    ) -> Tuple[List[float], List[float], List[float]]:

    for param in model.backbone.parameters():
        param.requires_grad = False

    optimizer = torch.optim.AdamW(model.step_head.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    model.train()
    loss_history = []
    acc_correct_hist = []
    acc_wrong_hist = []

    for epoch in range(num_epochs):
        epoch_losses = []
        correct_preds = 0; correct_total = 0
        wrong_preds = 0; wrong_total = 0

        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            step_positions = batch["step_positions"].to(device)
            step_labels = batch["step_labels"].to(device)
            step_mask = batch["step_mask"].to(device)

            # (batch_num, max_steps)
            step_scores = model(input_ids, attention_mask, step_positions)

            # scalar
            loss = prm_loss(step_scores, step_labels, step_mask)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.step_head.parameters(), 1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

            with torch.no_grad():
                # (batch_num, max_steps)
                pred_labels = (torch.sigmoid(step_scores) > 0.5).float()
                valid_mask = step_mask.bool()

                pos_mask = valid_mask & (step_labels == 1)
                if pos_mask.any():
                    correct_preds += (pred_labels[pos_mask] == 1).sum().item()
                    correct_total += pos_mask.sum().item()

                neg_mask = valid_mask & (step_labels == 0)
                if neg_mask.any():
                    wrong_preds += (pred_labels[neg_mask] == 0).sum().item()
                    wrong_total += neg_mask.sum().item()

        scheduler.step()
        mean_loss = sum(epoch_losses) / len(epoch_losses)
        acc_correct = correct_preds / max(correct_total, 1)
        acc_wrong = wrong_preds / max(wrong_total, 1)

        loss_history.append(mean_loss)
        acc_correct_hist.append(acc_correct)
        acc_wrong_hist.append(acc_wrong)

        print(f"  Epoch {epoch+1:3d}/{num_epochs} | Loss: {mean_loss:.4f} | Acc(correct steps): {acc_correct:.2%} | Acc(wrong steps): {acc_wrong:.2%}")

    return loss_history, acc_correct_hist, acc_wrong_hist

# == Test Run ===
prm_losses, prm_acc_c, prm_acc_w = train_prm(prm, prm_loader, num_epochs=2)


TRAINING: Process Reward Model (step-level BCE)
  Epoch   1/2 | Loss: 0.6891 | Acc(correct steps): 58.33% | Acc(wrong steps): 25.00%
  Epoch   2/2 | Loss: 0.6332 | Acc(correct steps): 100.00% | Acc(wrong steps): 0.00%


## 3.7 Inference

**PRM at Inference Time**

The PRM has two main uses at inference:

**1. Step-level scoring** (find where errors occur):
    Score each step and identify the first step that drops below a threshold.

**2. Best-of-N with PRM aggregation** (solution selection):
    Score each candidate solution and aggregate step scores.

Two common aggregation strategies:
- **Min-score**: r_solution = min(r_1, ..., r_K)
  Conservative: a solution is only as good as its weakest step.
  Used in Lightman et al. (2023).

- **Product**: r_solution = Π_{k=1}^K r_k
  Penalizes multiple low-confidence steps.
  Equivalent to summing log-probabilities.

- **Last-step**: r_solution = r_K
  Uses only the final step's score.
  Degenerates toward ORM behavior — generally avoided.

In [ ]:
@torch.no_grad()
def prm_score_steps(
        model: ProcessRewardModel, problem: str, steps: List[str], tokenizer, max_len: int = 512
    ) -> List[float]:
    """
    Score each step in a solution with the PRM.
    Returns a probability in [0, 1] for each step (1 = correct).
    """
    model.eval()

    sequence_parts = [problem]
    for step in steps:
        sequence_parts.append(step + STEP_SEPARATOR)
    full_text = " ".join(sequence_parts)

    # Output: (1, seq_len)
    encoded = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )

    # (1, seq_len)
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    sep_id = (input_ids[0] == tokenizer(" .", add_special_tokens=False)["input_ids"][0])
    step_pos_list = sep_id.nonzero(as_tuple=True)[0].tolist()[:len(steps)]

    if not step_pos_list:
        step_pos_list = list(range(1, len(steps) + 1))

    max_steps = len(steps)
    # (1, max_steps)
    step_positions = torch.zeros(1, max_steps, dtype=torch.long, device=device)
    for i, pos in enumerate(step_pos_list[:max_steps]):
        step_positions[0, i] = min(pos, max_len - 1)

    # (1, seq_len) → (1, max_steps)
    step_scores = model(input_ids, attention_mask, step_positions)

    # (1, max_steps) → list of floats
    step_probs = torch.sigmoid(step_scores[0]).cpu().tolist()[:len(steps)]
    return step_probs

def best_of_n_prm(
        model: ProcessRewardModel,
        problem: str,
        solution_steps: List[List[str]],
        tokenizer, aggregation: str = "min",
        verbose: bool = True
    ) -> Tuple[int, float, List[float]]:
    """
    Best-of-N selection using PRM scoring.
    Scores each candidate solution by aggregating its step scores, then returns the index of the best solution.
    """
    all_solution_scores = []

    for sol_idx, steps in enumerate(solution_steps):
        step_probs = prm_score_steps(model, problem, steps, tokenizer)

        if aggregation == "min":
            sol_score = min(step_probs) if step_probs else 0.0
        elif aggregation == "product":
            sol_score = math.prod(step_probs) if step_probs else 0.0
        elif aggregation == "mean":
            sol_score = sum(step_probs) / len(step_probs) if step_probs else 0.0
        elif aggregation == "last":
            sol_score = step_probs[-1] if step_probs else 0.0
        else:
            raise ValueError(f"Unknown aggregation: {aggregation}")

        all_solution_scores.append(sol_score)

        if verbose:
            print(f"  Solution {sol_idx+1}: aggregated={sol_score:.4f}")
            for k, (step, prob) in enumerate(zip(steps, step_probs)):
                flag = "❌" if prob < 0.5 else "✓"
                print(f"    Step {k+1}: p={prob:.3f} {flag} | {step[:50]}")

    best_idx = int(np.argmax(all_solution_scores))
    best_score = all_solution_scores[best_idx]
    return best_idx, best_score, all_solution_scores

# Demo PRM inference
solutions_as_steps = [
    [
        "Monday sales = 30% of 50 = 15.",
        "After Monday: 50 − 15 = 35.",
        "Tuesday sales = 20% of 35 = 7.",
        "After Tuesday: 35 − 7 = 28.",
    ],
    [
        "Monday sales = 30% of 50 = 15.",
        "Tuesday sales = 20% of 50 = 10.",   # ERROR
        "Total sold = 25.",
        "Remaining = 25.",
    ],
]

best_idx, best_score, all_scores = best_of_n_prm(
    prm, mc_problem, solutions_as_steps, tokenizer,
    aggregation="min", verbose=True
)
print(f"\nBest solution index: {best_idx+1} (score={best_score:.4f})")


DEMO: PRM Step Scoring and Best-of-N
  Solution 1: aggregated=0.5634
    Step 1: p=0.593 ✓ | Monday sales = 30% of 50 = 15.
    Step 2: p=0.563 ✓ | After Monday: 50 − 15 = 35.
    Step 3: p=0.587 ✓ | Tuesday sales = 20% of 35 = 7.
    Step 4: p=0.588 ✓ | After Tuesday: 35 − 7 = 28.
  Solution 2: aggregated=0.5699
    Step 1: p=0.590 ✓ | Monday sales = 30% of 50 = 15.
    Step 2: p=0.570 ✓ | Tuesday sales = 20% of 50 = 10.
    Step 3: p=0.583 ✓ | Total sold = 25.
    Step 4: p=0.582 ✓ | Remaining = 25.

Best solution index: 2 (score=0.5699)


# Summary

Note: A reward model by itself is just a scorer — it can evaluate reasoning steps, but it cannot update the base model's weights.

You need an optimization algorithm (e.g. GRPO) to actually improve the model.

## Summary Table

| Property                    | ORM                        | PRM                               |
|-----------------------------|----------------------------|------------------------------------|
| Granularity                 | Solution-level (sparse)    | Step-level (dense)                 |
| Training data               | (problem, correct, wrong)  | (problem, steps, step_labels)      |
| Annotation cost             | Low (auto-verify answer)   | High (human/MC/LLM per step)       |
| Training signal             | Sparse BT / BCE loss       | Dense per-step BCE loss            |
| Credit assignment           | No (all steps blamed)      | Yes (first error identified)       |
| BoN performance (large N)   | Good                       | Better (gap widens with N)         |
| RL reward type              | Sparse (terminal)          | Dense (per-step shaping)           |
| Scalability                 | High                       | Medium (annotation bottleneck)     |
| Error localization          | No                         | Yes                                |
| Risk of "reward hacking"    | Moderate                   | Higher (exploiting step scores)    |

## Key Limitations

**ORM limitations:**
1. Cannot identify WHERE in the reasoning chain an error occurred
2. Incorrect solutions with many correct steps get same reward as solutions that are wrong from step 1 — poor credit assignment
3. In RL, sparse terminal reward makes long-horizon training unstable

**PRM limitations:**
1. Annotation bottleneck: MC estimation is noisy, human annotation is expensive
2. "Reward hacking": models learn to score high on individual steps while still producing a wrong final answer
3. Qwen (2025) found many PRMs degrade to ORM-like behavior — they assign low scores to the FINAL step containing the answer, rather than the intermediate step where the error actually occurred
4. Step granularity is ambiguous — what counts as one "step"?
5. Difficult to apply to open-ended tasks (no clear "correct" label per step)